In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from keras.src.legacy.preprocessing.image import ImageDataGenerator
from PIL import Image
import pandas as pd
import os

Image.MAX_IMAGE_PIXELS = None

# Define paths
TRAIN_DIR = '/Users/sochea/Documents/MITE/dataset/training'
TEST_DIR = '/Users/sochea/Documents/MITE/dataset/testing'
CSV_PATH = '/Users/sochea/Documents/MITE/dataset/save_model/cnn_training_history.csv'
MODEL_PATH = '/Users/sochea/Documents/MITE/dataset/save_model/new_trained_model.h5'

# Parameters
img_size = 224
lr = 1e-4
num_classes = 2
num_epochs = 5


def check_images(directory):
    for root, _, files in os.walk(directory):
        for file in files:
            try:
                img_paths = os.path.join(root, file)
                img = Image.open(img_paths)
                img.verify()
            except (IOError, SyntaxError) as e:
                print(f"Corrupted or unreadable image: {img_paths}")


# Check training, validation, and test directories
check_images(TRAIN_DIR)
check_images(TEST_DIR)

# Data augmentation for training
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255.0,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=0.2
)

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(img_size, img_size),
    batch_size=64,
    class_mode='categorical',
    subset='training',
    color_mode='grayscale',
    seed=123,
    shuffle=True,
)

validation_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(img_size, img_size),
    batch_size=64,
    class_mode='categorical',
    subset='validation',
    color_mode='grayscale',
    seed=123,
    shuffle=True,
)

test_datagen = ImageDataGenerator(rescale=1.0 / 255.0)

test_generator = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=(img_size, img_size),
    batch_size=64,
    class_mode=None,
    color_mode='grayscale',
    shuffle=False
)

# Define the CNN model for binary classification with explicit Input layer
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(img_size, img_size, 1)),  # Specify the input shape here
    tf.keras.layers.Conv2D(32, (3, 3), padding='same', activation="relu"),
    tf.keras.layers.MaxPooling2D((2, 2), strides=2),

    tf.keras.layers.Conv2D(64, (3, 3), padding='same', activation="relu"),
    tf.keras.layers.MaxPooling2D((2, 2), strides=2),

    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(100, activation="relu"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dense(2, activation="softmax")  # Binary classification: harmful vs non-harmful
])

# Compile the model for binary classification
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
              loss='binary_crossentropy', metrics=['accuracy'])

# Train the model
history = model.fit(
    train_generator,
    epochs=num_epochs,
    validation_data=validation_generator,
)

# Save the trained model
model.save(MODEL_PATH)

# Save training history to CSV
pd.DataFrame(history.history).to_csv(CSV_PATH, index=False)


# Plot training and validation accuracy and loss
def plot_history(history_data):
    plt.figure(figsize=(12, 4))
    plt.subplot(1, 2, 1)
    plt.plot(history_data.history['accuracy'])
    plt.plot(history_data.history['val_accuracy'])
    plt.title('Model accuracy')
    plt.ylabel('Accuracy')
    plt.xlabel('Epoch')
    plt.legend(['Train', 'Validation'], loc='upper left')

    plt.subplot(1, 2, 2)
    plt.plot(history_data.history['loss'])
    plt.plot(history_data.history['val_loss'])
    plt.title('Model loss')
    plt.ylabel('Loss')
    plt.xlabel('Epoch')
    plt.legend(['Train', 'Validation'], loc='upper left')

    plt.show()


plot_history(history)


# Plot test images with predicted labels
def plot_test_images_with_labels(test_data_ex, prediction, num_images=5):
    images, _ = next(test_data_ex)  # Get the first batch of test images
    plt.figure(figsize=(12, 12))
    for i in range(num_images):
        img = images[i].reshape(img_size, img_size)
        predicted_label = np.argmax(prediction[i])
        label_text = 'harmful' if predicted_label == 0 else 'non_harmful'

        plt.subplot(3, 3, i + 1)
        plt.imshow(img, cmap='gray')
        plt.title(label_text)
        plt.axis('off')
    plt.tight_layout()
    plt.show()


# Make predictions on test data
predictions = model.predict(test_generator)

# Plot the first 5 test images with their predicted labels
plot_test_images_with_labels(test_generator, predictions, num_images=5)

Found 2177 images belonging to 2 classes.
Found 543 images belonging to 2 classes.
Found 143 images belonging to 1 classes.
Epoch 1/5


/opt/homebrew/lib/python3.11/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:122: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()
